# VisionPlate YOLOv8s Kaggle Training Notebook

Use this notebook to train the number plate detector on Kaggle Notebooks, save graphs for the report, and resume training if interrupted.

## 1. Install Requirements
Run once. Restart the kernel if packages were newly installed.

In [ ]:
!pip install ultralytics roboflow opencv-python matplotlib pandas seaborn

## 2. Check GPU

In [ ]:
# !pip uninstall torch torchvision torchaudio -y

In [ ]:
# !pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## 3. Download Dataset From Roboflow
Replace the values with your Roboflow dataset snippet. Choose YOLOv8 format.

In [ ]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="AJ4uKoH91ufHWzsNIvoG")
project = rf.workspace("abduls-workspace-qwhgu").project("indian-license-plate-merged")
version = project.version(2)
dataset = version.download("yolov8")
                
DATA_YAML = f'{dataset.location}/data.yaml'
print(DATA_YAML)

## 4. Train YOLOv8s
Start with `batch=4` on RTX 3050. If VRAM allows, increase to `batch=8`.

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8s.pt')
results = model.train(
    data=DATA_YAML,
    epochs=80,
    imgsz=640,
    batch=4,
    patience=15,
    optimizer='AdamW',
    lr0=0.001,
    cos_lr=True,
    degrees=8,
    translate=0.08,
    scale=0.4,
    shear=2,
    perspective=0.0005,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.08,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    project='/kaggle/working/runs',
    name='yolov8s_indian_plate'
)

## 5. Resume Training If Interrupted
Run this cell only if training stopped before completion.

In [ ]:
from ultralytics import YOLO

last_weights = '/kaggle/working/runs/yolov8s_indian_plate/weights/last.pt'
model = YOLO(last_weights)
model.train(resume=True)

## 6. Validate Best Model

In [ ]:
from ultralytics import YOLO

best_model = YOLO('/kaggle/working/runs/yolov8s_indian_plate/weights/best.pt')
metrics = best_model.val(data=DATA_YAML, imgsz=640)
print('mAP@0.5:', metrics.box.map50)
print('mAP@0.5:0.95:', metrics.box.map)

## 7. Run Sample Predictions
These generated images are useful as screenshots for your report.

In [ ]:
best_model.predict(
    source=f'{dataset.location}/test/images',
    imgsz=640,
    conf=0.25,
    save=True,
    project='/kaggle/working/runs',
    name='sample_predictions'
)

## 8. Display Training Graphs

In [ ]:
from IPython.display import Image, display
from pathlib import Path

run_dir = Path('/kaggle/working/runs/yolov8s_indian_plate')
for image_name in ['results.png', 'confusion_matrix.png', 'P_curve.png', 'R_curve.png', 'F1_curve.png', 'PR_curve.png']:
    image_path = run_dir / image_name
    if image_path.exists():
        print(image_name)
        display(Image(filename=str(image_path)))

## 9. Copy Best Model To Backend

In [ ]:
from pathlib import Path
import shutil

source = Path('/kaggle/working/runs/yolov8s_indian_plate/weights/best.pt')
destination = Path('/kaggle/working/best.pt')
destination.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(source, destination)
print('Copied to:', destination.resolve())